# Baseline ML Comparison for UAV Propeller Fault Diagnosis

This notebook evaluates classical machine learning baselines using the same physics-informed features and source-file-wise group validation protocol.

## Models included

1. Random Forest  
2. SVM  
3. XGBoost  
4. KNN  
5. Logistic Regression  

## Input file

```text
merged_physics.csv
```

The dataset must contain:

- `class_label`
- `source_file`
- measured numerical features
- physics-informed derived features

The notebook removes idle rows, generates non-overlapping windows, flattens each time-window for classical ML models, and evaluates models using group-wise validation to reduce source-file leakage.


In [1]:
# ============================================================
# Baseline ML Comparison for UAV Propeller Fault Diagnosis
# Models: Random Forest, SVM, XGBoost, KNN, Logistic Regression
# Validation: Strict run-wise split and Stratified Group K-Fold
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not available. Install using: pip install xgboost")
    print("XGBoost error:", e)

# --------------------------
# Global settings
# --------------------------
SEED = 42
np.random.seed(SEED)

DATA_FILE = "merged_physics.csv"
OUTPUT_DIR = Path("Results_Baseline_ML_Groupwise")

WINDOW_SIZE = 50
STRIDE = 50
FILTER_ACTIVE_REGION = True
ACTIVE_RPM_MIN = 500
ACTIVE_ESC_MIN = 1100

TEST_SIZE = 0.33
N_SPLITS = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["legend.fontsize"] = 12
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

print("Configuration loaded.")
print("DATA_FILE:", DATA_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


Configuration loaded.
DATA_FILE: merged_physics.csv
OUTPUT_DIR: Results_Baseline_ML_Groupwise


In [2]:
# ============================================================
# Utility functions
# ============================================================

def find_col(df, possible_names, required=True):
    """
    Find a column by exact or partial matching.
    """
    cols = list(df.columns)
    lower_map = {c.lower().strip(): c for c in cols}

    for name in possible_names:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]

    for c in cols:
        c_low = c.lower().strip()
        for name in possible_names:
            if name.lower().strip() in c_low:
                return c

    if required:
        raise ValueError(f"Could not find any of these columns: {possible_names}")
    return None


def majority_label(labels):
    """
    Return the majority label in a sequence.
    """
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


def compute_metrics(y_true, y_pred, all_labels):
    """
    Compute macro and weighted classification metrics.
    """
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
    }


def save_confusion_matrix(y_true, y_pred, labels, class_names, out_pdf, title):
    """
    Save confusion matrix as PDF.
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    plt.tight_layout()
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close(fig)


def save_metric_bar(metrics_df, out_pdf, title):
    """
    Save bar plot of core metrics as PDF.
    """
    metric_cols = ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]
    plot_df = metrics_df.set_index("Model")[metric_cols]

    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    plot_df.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score", fontweight="bold")
    ax.set_xlabel("Model", fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.legend(frameon=True)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close(fig)


print("Utility functions ready.")


Utility functions ready.


In [3]:
# ============================================================
# Load dataset and select features
# ============================================================

df = pd.read_csv(DATA_FILE)
df.columns = df.columns.str.strip()

label_col = find_col(df, ["class_label", "label", "class"])
group_col = find_col(df, ["source_file", "source", "file", "run", "group"])

rpm_col = find_col(df, ["Motor Electrical Speed (RPM)", "RPM", "Motor Speed"], required=False)
esc_col = find_col(df, ["ESC signal (µs)", "ESC signal", "ESC"], required=False)

df[label_col] = df[label_col].astype(str).str.strip()
df[group_col] = df[group_col].astype(str).str.strip()

# Convert object-like numeric columns where possible
for col in df.columns:
    if col not in [label_col, group_col]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove idle region
if FILTER_ACTIVE_REGION:
    if rpm_col is not None:
        df = df[df[rpm_col] > ACTIVE_RPM_MIN].copy()
        print(f"Active-region filtering applied: {rpm_col} > {ACTIVE_RPM_MIN}")
    elif esc_col is not None:
        df = df[df[esc_col] >= ACTIVE_ESC_MIN].copy()
        print(f"Active-region filtering applied: {esc_col} >= {ACTIVE_ESC_MIN}")
    else:
        print("Warning: No RPM or ESC column found. Active-region filtering skipped.")

# Select numerical feature columns
exclude_cols = [label_col, group_col]
feature_cols = [
    c for c in df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])
]

# Drop rows with missing values in selected features or labels/groups
df = df.dropna(subset=feature_cols + [label_col, group_col]).copy()

print("Dataset shape after cleaning:", df.shape)
print("Label column:", label_col)
print("Group column:", group_col)
print("Number of numerical features:", len(feature_cols))
print("Class distribution:")
print(df[label_col].value_counts())
print("\nSource-file groups:", df[group_col].nunique())


Active-region filtering applied: Motor Electrical Speed (RPM) > 500
Dataset shape after cleaning: (1396, 33)
Label column: class_label
Group column: source_file
Number of numerical features: 31
Class distribution:
class_label
bend      591
normal    589
crack     216
Name: count, dtype: int64

Source-file groups: 9


In [4]:
# ============================================================
# Generate non-overlapping windows within each source file
# ============================================================

def create_windows_from_dataframe(df, feature_cols, label_col, group_col, window_size=50, stride=50):
    X_windows = []
    y_windows = []
    group_windows = []
    start_indices = []

    for group_name, gdf in df.groupby(group_col, sort=False):
        gdf = gdf.reset_index(drop=True)

        X_arr = gdf[feature_cols].values.astype(np.float32)
        y_arr = gdf[label_col].values

        n = len(gdf)
        if n < window_size:
            continue

        for start in range(0, n - window_size + 1, stride):
            end = start + window_size
            X_win = X_arr[start:end]
            y_win = majority_label(y_arr[start:end])

            X_windows.append(X_win)
            y_windows.append(y_win)
            group_windows.append(group_name)
            start_indices.append(start)

    X_windows = np.asarray(X_windows, dtype=np.float32)
    y_windows = np.asarray(y_windows)
    group_windows = np.asarray(group_windows)
    start_indices = np.asarray(start_indices)

    return X_windows, y_windows, group_windows, start_indices


X_seq, y_str, groups, start_indices = create_windows_from_dataframe(
    df=df,
    feature_cols=feature_cols,
    label_col=label_col,
    group_col=group_col,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

if X_seq.size == 0:
    raise ValueError("No windows were generated. Reduce WINDOW_SIZE or check source_file lengths.")

print("Sequential window shape:", X_seq.shape)
print("Number of windows:", len(y_str))
print("Window class distribution:")
print(pd.Series(y_str).value_counts())
print("\nWindow group count:", len(np.unique(groups)))

# Flatten windows for classical ML models
n_windows, t_steps, n_features = X_seq.shape
X_flat = X_seq.reshape(n_windows, t_steps * n_features)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_str)

class_names = label_encoder.classes_
all_labels = np.arange(len(class_names))

print("\nFlattened feature shape for ML baselines:", X_flat.shape)
print("Classes:", list(class_names))


Sequential window shape: (21, 50, 31)
Number of windows: 21
Window class distribution:
bend      9
normal    9
crack     3
Name: count, dtype: int64

Window group count: 9

Flattened feature shape for ML baselines: (21, 1550)
Classes: [np.str_('bend'), np.str_('crack'), np.str_('normal')]


In [5]:
# ============================================================
# Define baseline ML models
# ============================================================

models = {
    "Random_Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1
    ),

    "SVM_RBF": SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=SEED
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        metric="minkowski"
    ),

    "Logistic_Regression": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )

print("Models included:")
for name in models:
    print("-", name)


Models included:
- Random_Forest
- SVM_RBF
- KNN
- Logistic_Regression
- XGBoost


In [6]:
# ============================================================
# Strict source-file-wise train/test split
# ============================================================

def strict_group_split(X, y, groups, test_size=0.33, random_state=42):
    """
    Split complete source-file groups into train and test sets.
    Attempts stratification using majority class of each group.
    Falls back to GroupShuffleSplit if stratification is not feasible.
    """
    unique_groups = np.unique(groups)

    group_majority = []
    for g in unique_groups:
        group_majority.append(majority_label(y[groups == g]))
    group_majority = np.asarray(group_majority)

    try:
        train_groups, test_groups = train_test_split(
            unique_groups,
            test_size=test_size,
            random_state=random_state,
            stratify=group_majority
        )
    except Exception as e:
        print("Stratified file-level split failed. Using GroupShuffleSplit.")
        print("Reason:", e)
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx_tmp, test_idx_tmp = next(splitter.split(X, y, groups))
        train_groups = np.unique(groups[train_idx_tmp])
        test_groups = np.unique(groups[test_idx_tmp])

    train_mask = np.isin(groups, train_groups)
    test_mask = np.isin(groups, test_groups)

    return np.where(train_mask)[0], np.where(test_mask)[0], train_groups, test_groups


train_idx, test_idx, train_groups, test_groups = strict_group_split(
    X_flat, y, groups, test_size=TEST_SIZE, random_state=SEED
)

X_train, X_test = X_flat[train_idx], X_flat[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

split_info = {
    "train_groups": train_groups.tolist(),
    "test_groups": test_groups.tolist(),
    "n_train_windows": int(len(train_idx)),
    "n_test_windows": int(len(test_idx))
}

strict_dir = OUTPUT_DIR / "Strict_Runwise"
strict_dir.mkdir(parents=True, exist_ok=True)

with open(strict_dir / "strict_split_source_files.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=4)

print("Strict group-wise split completed.")
print("Train windows:", len(train_idx))
print("Test windows:", len(test_idx))
print("Train groups:", train_groups)
print("Test groups:", test_groups)
print("\nTrain class distribution:")
print(pd.Series(y_train).map(dict(enumerate(class_names))).value_counts())
print("\nTest class distribution:")
print(pd.Series(y_test).map(dict(enumerate(class_names))).value_counts())


Strict group-wise split completed.
Train windows: 14
Test windows: 7
Train groups: ['Propeller_Data_1045_crack3.csv' 'Propeller_Data_1045_bend1.csv'
 'Propeller_Data_1045_bend2.csv' 'Propeller_Data_1045_crack2.csv'
 'Propeller_Data_1045_normal2.csv' 'Propeller_Data_1045_normal1.csv']
Test groups: ['Propeller_Data_1045_crack1.csv' 'Propeller_Data_1045_bend3.csv'
 'Propeller_Data_1045_normal3.csv']

Train class distribution:
bend      6
normal    6
crack     2
Name: count, dtype: int64

Test class distribution:
bend      3
normal    3
crack     1
Name: count, dtype: int64


In [7]:
# ============================================================
# Run strict source-file-wise baseline comparison
# ============================================================

strict_results = []
strict_predictions = []

for model_name, clf in models.items():
    print(f"\nTraining: {model_name}")

    model_dir = strict_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    metrics = compute_metrics(y_test, y_pred, all_labels)
    metrics["Model"] = model_name
    strict_results.append(metrics)

    report = classification_report(
        y_test,
        y_pred,
        labels=all_labels,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    with open(model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)

    save_confusion_matrix(
        y_true=y_test,
        y_pred=y_pred,
        labels=all_labels,
        class_names=class_names,
        out_pdf=model_dir / "confusion_matrix.pdf",
        title=f"{model_name} - Strict Run-Wise"
    )

    pred_df = pd.DataFrame({
        "group": groups[test_idx],
        "true_label": label_encoder.inverse_transform(y_test),
        "predicted_label": label_encoder.inverse_transform(y_pred)
    })
    pred_df.to_csv(model_dir / "predictions.csv", index=False)

    for g, yt, yp in zip(groups[test_idx], y_test, y_pred):
        strict_predictions.append({
            "Model": model_name,
            "group": g,
            "true_label": label_encoder.inverse_transform([yt])[0],
            "predicted_label": label_encoder.inverse_transform([yp])[0]
        })

strict_metrics_df = pd.DataFrame(strict_results)
strict_metrics_df = strict_metrics_df[
    ["Model", "Accuracy", "Precision_macro", "Recall_macro", "F1_macro",
     "Precision_weighted", "Recall_weighted", "F1_weighted"]
]

strict_metrics_df.to_csv(strict_dir / "baseline_strict_runwise_metrics.csv", index=False)
pd.DataFrame(strict_predictions).to_csv(strict_dir / "baseline_strict_runwise_predictions_all_models.csv", index=False)

save_metric_bar(
    strict_metrics_df,
    strict_dir / "baseline_strict_runwise_metrics_bar.pdf",
    "Baseline ML Models - Strict Run-Wise Validation"
)

strict_metrics_df



Training: Random_Forest

Training: SVM_RBF

Training: KNN

Training: Logistic_Regression

Training: XGBoost


,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted
0,Random_Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,SVM_RBF,0.857143,0.583333,0.666667,0.619048,0.750000,0.857143,0.795918
2,KNN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,Logistic_Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,XGBoost,0.714286,0.555556,0.555556,0.555556,0.714286,0.714286,0.714286


In [8]:
# ============================================================
# Stratified Group K-Fold baseline comparison
# ============================================================

kfold_dir = OUTPUT_DIR / "Stratified_Group_KFold"
kfold_dir.mkdir(parents=True, exist_ok=True)

n_unique_groups = len(np.unique(groups))
n_splits = min(N_SPLITS, n_unique_groups)

if n_splits < 2:
    raise ValueError("At least two source-file groups are required for Group K-Fold validation.")

print("Number of unique groups:", n_unique_groups)
print("Number of folds:", n_splits)

sgkf = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

fold_results = []
fold_predictions = []

for fold_id, (train_idx, test_idx) in enumerate(sgkf.split(X_flat, y, groups), start=1):
    print(f"\n========== Fold {fold_id}/{n_splits} ==========")

    fold_dir = kfold_dir / f"Fold_{fold_id}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    X_train_fold, X_test_fold = X_flat[train_idx], X_flat[test_idx]
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]

    train_groups_fold = np.unique(groups[train_idx])
    test_groups_fold = np.unique(groups[test_idx])

    fold_split_info = {
        "fold": fold_id,
        "train_groups": train_groups_fold.tolist(),
        "test_groups": test_groups_fold.tolist(),
        "n_train_windows": int(len(train_idx)),
        "n_test_windows": int(len(test_idx))
    }

    with open(fold_dir / "fold_source_files.json", "w", encoding="utf-8") as f:
        json.dump(fold_split_info, f, indent=4)

    print("Train windows:", len(train_idx), "Test windows:", len(test_idx))
    print("Test groups:", test_groups_fold)

    for model_name, clf in models.items():
        print(f"Training {model_name}...")

        model_dir = fold_dir / model_name
        model_dir.mkdir(parents=True, exist_ok=True)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", clf)
        ])

        pipe.fit(X_train_fold, y_train_fold)
        y_pred_fold = pipe.predict(X_test_fold)

        metrics = compute_metrics(y_test_fold, y_pred_fold, all_labels)
        metrics["Model"] = model_name
        metrics["Fold"] = fold_id
        fold_results.append(metrics)

        report = classification_report(
            y_test_fold,
            y_pred_fold,
            labels=all_labels,
            target_names=class_names,
            digits=4,
            zero_division=0
        )

        with open(model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
            f.write(report)

        save_confusion_matrix(
            y_true=y_test_fold,
            y_pred=y_pred_fold,
            labels=all_labels,
            class_names=class_names,
            out_pdf=model_dir / "confusion_matrix.pdf",
            title=f"{model_name} - Fold {fold_id}"
        )

        pred_df = pd.DataFrame({
            "fold": fold_id,
            "group": groups[test_idx],
            "true_label": label_encoder.inverse_transform(y_test_fold),
            "predicted_label": label_encoder.inverse_transform(y_pred_fold)
        })
        pred_df.to_csv(model_dir / "predictions.csv", index=False)

        for g, yt, yp in zip(groups[test_idx], y_test_fold, y_pred_fold):
            fold_predictions.append({
                "Fold": fold_id,
                "Model": model_name,
                "group": g,
                "true_label": label_encoder.inverse_transform([yt])[0],
                "predicted_label": label_encoder.inverse_transform([yp])[0]
            })

fold_results_df = pd.DataFrame(fold_results)
fold_results_df = fold_results_df[
    ["Fold", "Model", "Accuracy", "Precision_macro", "Recall_macro", "F1_macro",
     "Precision_weighted", "Recall_weighted", "F1_weighted"]
]

fold_results_df.to_csv(kfold_dir / "baseline_group_kfold_fold_metrics.csv", index=False)
pd.DataFrame(fold_predictions).to_csv(kfold_dir / "baseline_group_kfold_predictions_all_models.csv", index=False)

fold_results_df


Number of unique groups: 9
Number of folds: 3

========== Fold 1/3 ==========
Train windows: 14 Test windows: 7
Test groups: ['Propeller_Data_1045_bend1.csv' 'Propeller_Data_1045_bend3.csv'
 'Propeller_Data_1045_crack3.csv']
Training Random_Forest...
Training SVM_RBF...
Training KNN...
Training Logistic_Regression...
Training XGBoost...

========== Fold 2/3 ==========
Train windows: 14 Test windows: 7
Test groups: ['Propeller_Data_1045_bend2.csv' 'Propeller_Data_1045_crack2.csv'
 'Propeller_Data_1045_normal1.csv']
Training Random_Forest...
Training SVM_RBF...
Training KNN...
Training Logistic_Regression...
Training XGBoost...

========== Fold 3/3 ==========
Train windows: 14 Test windows: 7
Test groups: ['Propeller_Data_1045_crack1.csv' 'Propeller_Data_1045_normal2.csv'
 'Propeller_Data_1045_normal3.csv']
Training Random_Forest...
Training SVM_RBF...
Training KNN...
Training Logistic_Regression...
Training XGBoost...


,Fold,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted
0,1,Random_Forest,0.714286,0.666667,0.555556,0.600000,1.000000,0.714286,0.828571
1,1,SVM_RBF,0.571429,0.666667,0.500000,0.555556,1.000000,0.571429,0.714286
2,1,KNN,0.142857,0.166667,0.333333,0.222222,0.071429,0.142857,0.095238
3,1,Logistic_Regression,0.714286,0.666667,0.555556,0.600000,1.000000,0.714286,0.828571
4,1,XGBoost,0.857143,0.666667,0.611111,0.636364,1.000000,0.857143,0.922078
5,2,Random_Forest,0.857143,0.583333,0.666667,0.619048,0.750000,0.857143,0.795918
6,2,SVM_RBF,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,2,KNN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,2,Logistic_Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,2,XGBoost,0.857143,0.583333,0.666667,0.619048,0.750000,0.857143,0.795918


In [9]:
# ============================================================
# Summarize Stratified Group K-Fold results: mean and std
# ============================================================

summary_rows = []

metric_cols = [
    "Accuracy",
    "Precision_macro",
    "Recall_macro",
    "F1_macro",
    "Precision_weighted",
    "Recall_weighted",
    "F1_weighted"
]

for model_name, gdf in fold_results_df.groupby("Model"):
    row = {"Model": model_name}
    for metric in metric_cols:
        row[f"{metric}_Mean"] = gdf[metric].mean()
        row[f"{metric}_Std"] = gdf[metric].std(ddof=1) if len(gdf) > 1 else 0.0
    summary_rows.append(row)

kfold_summary_df = pd.DataFrame(summary_rows)

# Sort by macro F1 mean
kfold_summary_df = kfold_summary_df.sort_values("F1_macro_Mean", ascending=False)

kfold_summary_df.to_csv(kfold_dir / "baseline_group_kfold_mean_std_summary.csv", index=False)

# Bar plot for mean metrics
bar_df = kfold_summary_df.rename(columns={
    "Accuracy_Mean": "Accuracy",
    "Precision_macro_Mean": "Precision_macro",
    "Recall_macro_Mean": "Recall_macro",
    "F1_macro_Mean": "F1_macro"
})[["Model", "Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]]

save_metric_bar(
    bar_df,
    kfold_dir / "baseline_group_kfold_mean_metrics_bar.pdf",
    "Baseline ML Models - Stratified Group K-Fold Mean Metrics"
)

kfold_summary_df


,Model,Accuracy_Mean,Accuracy_Std,Precision_macro_Mean,Precision_macro_Std,Recall_macro_Mean,Recall_macro_Std,F1_macro_Mean,F1_macro_Std,Precision_weighted_Mean,Precision_weighted_Std,Recall_weighted_Mean,Recall_weighted_Std,F1_weighted_Mean,F1_weighted_Std
1,Logistic_Regression,0.904762,0.164957,0.777778,0.192450,0.740741,0.231296,0.755556,0.214303,1.000000,0.000000,0.904762,0.164957,0.942857,0.098974
2,Random_Forest,0.857143,0.142857,0.638889,0.048113,0.629630,0.064150,0.628571,0.034339,0.916667,0.144338,0.857143,0.142857,0.874830,0.109623
3,SVM_RBF,0.714286,0.247436,0.666667,0.333333,0.574074,0.394144,0.607407,0.369406,0.952381,0.082479,0.714286,0.247436,0.800000,0.173793
4,XGBoost,0.714286,0.247436,0.500000,0.220479,0.481481,0.274049,0.485137,0.247088,0.797619,0.183271,0.714286,0.247436,0.744094,0.208777
0,KNN,0.380952,0.540848,0.388889,0.535758,0.444444,0.509175,0.407407,0.525091,0.357143,0.557875,0.380952,0.540848,0.365079,0.551916


In [10]:
# ============================================================
# Generate LaTeX table for paper
# ============================================================

def dataframe_to_latex_baseline_table(df_summary, caption, label):
    """
    Create compact LaTeX table for mean and standard deviation metrics.
    """
    temp = df_summary.copy()

    lines = []
    lines.append(r"\begin{table*}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\scriptsize")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(r"\begin{tabular}{lcccc}")
    lines.append(r"\hline")
    lines.append(r"\textbf{Model} & \textbf{Accuracy} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\")
    lines.append(r"\hline")

    for _, row in temp.iterrows():
        model = row["Model"].replace("_", r"\_")
        acc = f"{row['Accuracy_Mean']:.3f} $\\pm$ {row['Accuracy_Std']:.3f}"
        prec = f"{row['Precision_macro_Mean']:.3f} $\\pm$ {row['Precision_macro_Std']:.3f}"
        rec = f"{row['Recall_macro_Mean']:.3f} $\\pm$ {row['Recall_macro_Std']:.3f}"
        f1 = f"{row['F1_macro_Mean']:.3f} $\\pm$ {row['F1_macro_Std']:.3f}"
        lines.append(f"{model} & {acc} & {prec} & {rec} & {f1} \\\\")

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\end{table*}")

    return "\n".join(lines)


latex_table = dataframe_to_latex_baseline_table(
    kfold_summary_df,
    caption="Baseline machine learning comparison using Stratified Group K-Fold validation.",
    label="tab:baseline_ml_group_kfold"
)

with open(kfold_dir / "baseline_group_kfold_latex_table.txt", "w", encoding="utf-8") as f:
    f.write(latex_table)

print(latex_table)


\begin{table*}[htbp]
\centering
\scriptsize
\caption{Baseline machine learning comparison using Stratified Group K-Fold validation.}
\label{tab:baseline_ml_group_kfold}
\resizebox{\textwidth}{!}{
\begin{tabular}{lcccc}
\hline
\textbf{Model} & \textbf{Accuracy} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
Logistic\_Regression & 0.905 $\pm$ 0.165 & 0.778 $\pm$ 0.192 & 0.741 $\pm$ 0.231 & 0.756 $\pm$ 0.214 \\
Random\_Forest & 0.857 $\pm$ 0.143 & 0.639 $\pm$ 0.048 & 0.630 $\pm$ 0.064 & 0.629 $\pm$ 0.034 \\
SVM\_RBF & 0.714 $\pm$ 0.247 & 0.667 $\pm$ 0.333 & 0.574 $\pm$ 0.394 & 0.607 $\pm$ 0.369 \\
XGBoost & 0.714 $\pm$ 0.247 & 0.500 $\pm$ 0.220 & 0.481 $\pm$ 0.274 & 0.485 $\pm$ 0.247 \\
KNN & 0.381 $\pm$ 0.541 & 0.389 $\pm$ 0.536 & 0.444 $\pm$ 0.509 & 0.407 $\pm$ 0.525 \\
\hline
\end{tabular}
}
\end{table*}


## Recommended manuscript text

You may add the following paragraph before the baseline table in the paper:

```latex
\paragraph{Baseline Machine Learning Comparison}
To evaluate whether deep temporal models provide additional benefit over classical machine learning methods, five baseline classifiers were implemented using the same measured and physics-informed features. The time-windowed samples were flattened into fixed-length feature vectors, and Random Forest, SVM, XGBoost, KNN, and Logistic Regression were evaluated using the same source-file-wise group validation protocol. This comparison ensures that all baseline models are tested under the same leakage-controlled setting as the proposed deep learning framework.
```


In [11]:
# ============================================================
# Label-Shuffle Leakage Test for Baseline ML Models
# Models: Random Forest, SVM, XGBoost, KNN, Logistic Regression
# Split: Same strict source-file-wise split
# ============================================================

label_shuffle_dir = OUTPUT_DIR / "Label_Shuffle_Test"
label_shuffle_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
y_train_shuffled = y_train.copy()
rng.shuffle(y_train_shuffled)

chance_level = 1.0 / len(class_names)

label_shuffle_results = []
label_shuffle_predictions = []

for model_name, clf in models.items():
    print(f"\nTraining with shuffled labels: {model_name}")

    model_dir = label_shuffle_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])

    pipe.fit(X_train, y_train_shuffled)
    y_pred = pipe.predict(X_test)

    metrics = compute_metrics(y_test, y_pred, all_labels)
    metrics["Model"] = model_name
    metrics["Expected_Chance_Level"] = chance_level
    label_shuffle_results.append(metrics)

    report = classification_report(
        y_test,
        y_pred,
        labels=all_labels,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    with open(model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write("Label-Shuffle Leakage Test\n")
        f.write(f"Expected chance-level accuracy: {chance_level:.4f}\n\n")
        f.write(report)

    save_confusion_matrix(
        y_true=y_test,
        y_pred=y_pred,
        labels=all_labels,
        class_names=class_names,
        out_pdf=model_dir / "confusion_matrix.pdf",
        title=f"{model_name} - Label-Shuffle Test"
    )

    pred_df = pd.DataFrame({
        "group": groups[test_idx],
        "true_label": label_encoder.inverse_transform(y_test),
        "predicted_label": label_encoder.inverse_transform(y_pred)
    })

    pred_df.to_csv(model_dir / "predictions.csv", index=False)

label_shuffle_metrics_df = pd.DataFrame(label_shuffle_results)

label_shuffle_metrics_df = label_shuffle_metrics_df[
    [
        "Model",
        "Expected_Chance_Level",
        "Accuracy",
        "Precision_macro",
        "Recall_macro",
        "F1_macro",
        "Precision_weighted",
        "Recall_weighted",
        "F1_weighted"
    ]
]

label_shuffle_metrics_df.to_csv(
    label_shuffle_dir / "baseline_label_shuffle_metrics.csv",
    index=False
)

save_metric_bar(
    label_shuffle_metrics_df[["Model", "Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]],
    label_shuffle_dir / "baseline_label_shuffle_metrics_bar.pdf",
    "Baseline ML Models - Label-Shuffle Leakage Test"
)

label_shuffle_metrics_df


Training with shuffled labels: Random_Forest

Training with shuffled labels: SVM_RBF

Training with shuffled labels: KNN

Training with shuffled labels: Logistic_Regression

Training with shuffled labels: XGBoost


,Model,Expected_Chance_Level,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted
0,Random_Forest,0.333333,0.285714,0.194444,0.222222,0.206349,0.250000,0.285714,0.265306
1,SVM_RBF,0.333333,0.571429,0.388889,0.444444,0.412698,0.500000,0.571429,0.530612
2,KNN,0.333333,0.428571,0.300000,0.333333,0.300000,0.385714,0.428571,0.385714
3,Logistic_Regression,0.333333,0.428571,0.300000,0.333333,0.300000,0.385714,0.428571,0.385714
4,XGBoost,0.333333,0.571429,0.366667,0.444444,0.383333,0.471429,0.571429,0.492857
